# 04 Generalization: per-attack EER on ASVspoof eval

Run only after `hypotheses.md` was frozen (commit 1d896cd). Each final model is scored on eval once.
Per-attack EER = that attack's fakes against all eval real clips (`deepvoice.metrics.per_attack_eer`).
A16 and A19 reuse training algorithms A04 and A06 ("repeated"); the other 11 are unseen.

**Rung 0** (silence/duration only). The rung 0 model is the one with the lower dev EER: the
depth-3 tree (dev 28.0% vs 30.4% for logistic regression). It is refit here deterministically
(seed 0, identical to `results/rung0.csv`).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.data import REPEATED_ATTACKS
from deepvoice.metrics import per_attack_eer
from deepvoice.models import build

p = pd.read_parquet("data/protocols.parquet")
S = {s: pd.read_parquet(f"data/features/silence_{s}.parquet").merge(p[["file", "label", "speaker", "attack"]], on="file")
     for s in ["train", "eval"]}
F = ["lead_sil", "trail_sil", "sil_ratio", "duration"]
rows = {}
for m, params in [("tree", {"max_depth": 3}), ("logreg", {"C": 1.0})]:
    model = build(m, 0, **params).fit(S["train"][F].to_numpy(), S["train"].label.to_numpy())
    s = model.predict_proba(S["eval"][F].to_numpy())[:, 1]
    rows[m] = per_attack_eer(S["eval"].label, s, S["eval"].attack)
pa = pd.DataFrame(rows).mul(100)
pa["type"] = ["repeated" if a in REPEATED_ATTACKS else "unseen" for a in pa.index]
pa.to_csv(ROOT / "results/rung0_eval_per_attack.csv", float_format="%.2f")
r0 = pd.read_csv(ROOT / "results/rung0.csv").query("split == 'eval'").set_index("model")
pa.round(1)

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(figsize=(6.4, 3.2))
d = pa["tree"]
x = range(len(d))
bars = ax.bar(x, d, color=["#c3c2b7" if t == "repeated" else "#52514e" for t in pa.type], width=0.7)
for b, t in zip(bars, pa.type):
    if t == "repeated":
        b.set_hatch("////"); b.set_edgecolor("white")
for i, v in zip(x, d):
    ax.text(i, v + 1, f"{v:.0f}", ha="center", fontsize=7.5, color=INK)
ax.axhline(50, color=MUTED, ls=":", lw=1); ax.text(-0.4, 51, "chance (50%)", fontsize=8, color=MUTED)
ax.axhline(100 * r0.loc["tree", "eer"], color="#e34948", ls="--", lw=1.2)
ax.text(-0.4, 100 * r0.loc["tree", "eer"] + 1.5, f"pooled eval EER {100 * r0.loc['tree', 'eer']:.1f}%", fontsize=8, color="#e34948")
ax.set_xticks(list(x), d.index); ax.set_ylim(0, 60)
ax.set_ylabel("eval EER % (silence/duration only)")
from matplotlib.patches import Patch

ax.legend(handles=[Patch(color="#52514e", label="unseen attack"), Patch(facecolor="#c3c2b7", hatch="////", edgecolor="white", label="repeated (A16=A04, A19=A06)")],
          frameon=False, loc="upper left", bbox_to_anchor=(0, 0.72), fontsize=8)
ax.set_title("Rung 0 on ASVspoof eval, per attack", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/rung0_eval_per_attack.png", dpi=200)
fig.savefig(ROOT / "results/figures/rung0_eval_per_attack.pdf")

## E4: the three final models on In-the-Wild (H3, H4)

The final model of each rung (chosen on dev, see notebooks 02/03 and `hypotheses.md`) is refit here
from code with its saved settings (deterministic, seed 0) and scored on train, dev, eval and
In-the-Wild (ITW). The eval numbers must equal the ones already recorded; this re-runs the same
models, it is not a new look at eval.

- **H3 (ITW part):** the rung 2 minus rung 1 margin is reported on eval and ITW; a shrinkage is
  only claimed if it is >= 10 pp.
- **H4:** on ITW, error at the dev threshold (mean of false-accept and false-reject rate) minus the
  best-threshold error (ITW EER) >= 10 pp for at least 2 of the 3 rungs.
- Rung 0 on ITW is the shortcut control: if silence alone separates ITW, ITW has a shortcut too.

In [ ]:
import json

import numpy as np

from deepvoice.features import load_xy
from deepvoice.metrics import eer, error_at_threshold
from deepvoice.models import run
from deepvoice.xlsr import LAYERS

SPLITS = ["train", "dev", "eval", "itw"]
DATASET = {"train": "asvspoof", "dev": "asvspoof", "eval": "asvspoof", "itw": "itw"}


def xlsr_frames(layer):
    k, cols, out = LAYERS.index(layer), [f"x{i:04d}" for i in range(1024)], {}
    for s in SPLITS:
        meta = load_xy(s, groups=())
        assert Path(f"data/features/xlsr300m_{s}_files.txt").read_text().split() == list(meta.file)
        e = np.load(f"data/features/xlsr300m_{s}.npy", mmap_mode="r")
        out[s] = pd.concat([meta, pd.DataFrame(np.asarray(e[:, k], dtype=np.float32), columns=cols)], axis=1)
    return out, cols


gap = pd.read_csv(ROOT / "results/e3_gap.csv")
hgb_params = json.loads(gap[(gap.model == "hgb") & (gap["selected by"] == "dev")].params.iloc[0])
lfcc = {s: load_xy(s, groups=("lfcc",), n_coef=40, deltas=2, stats=("mean", "std")) for s in SPLITS}
sil = {s: load_xy(s, groups=("silence",)) for s in SPLITS}
xl, xcols = xlsr_frames(3)
FINAL = [  # rung, model, data, feature columns, params, experiment, feature_set
    (0, "tree", sil, ["lead_sil", "trail_sil", "sil_ratio", "duration"], {"max_depth": 3}, "rung0", None),
    (1, "hgb", lfcc, list(lfcc["train"].columns[4:]), hgb_params, "rung1", "lfcc40dd_e3devsel"),
    (2, "logreg", xl, xcols, {"C": 1.0}, "rung2", "xlsr300m_L03"),
]

In [ ]:
rows, scores = [], {}
for rung, name, D, X, params, exp, fs in FINAL:
    config = f"{exp}_{name}_{fs}_s0" if fs else f"{exp}_{name}_s0"
    before = pd.read_csv(ROOT / f"results/{exp}.csv").query("config == @config and split == 'eval'").eer
    model, res = run(rung, name, X, D["train"], [(DATASET[s], s, D[s]) for s in SPLITS],
                     params=params, experiment=exp, feature_set=fs)
    after = {r["split"]: r for r in res}  # results CSVs keep 6 significant digits
    assert len(before) == 1 and np.isclose(before.iloc[0], after["eval"]["eer"], rtol=1e-5, atol=0), "eval not reproduced"
    sc = {s: model.predict_proba(D[s][X].to_numpy())[:, 1] for s in SPLITS}
    scores[rung] = sc
    thr = eer(D["dev"].label, sc["dev"])[1]
    row = {"rung": rung, "model": f"{name}:{fs}" if fs else name, "dev_threshold": thr}
    for s in ["dev", "eval", "itw"]:
        row[f"{s}_eer"], row[f"{s}_ci"] = after[s]["eer"], f"{100*after[s]['ci_low']:.1f}-{100*after[s]['ci_high']:.1f}"
        row[f"{s}_err_at_dev_thr"] = error_at_threshold(D[s].label, sc[s], thr)
    rows.append(row)
e4 = pd.DataFrame(rows)
e4.to_csv(ROOT / "results/e4_itw.csv", index=False, float_format="%.6g")
show = e4.copy()
for c in [c for c in e4.columns if c.endswith("_eer") or c.endswith("_thr") and c != "dev_threshold"]:
    show[c] = (100 * show[c]).round(2)
show

## Hypothesis verdicts from E4 (rules from the frozen `hypotheses.md`)

In [ ]:
by = e4.set_index("rung")
m_eval = by.loc[1, "eval_eer"] - by.loc[2, "eval_eer"]
m_itw = by.loc[1, "itw_eer"] - by.loc[2, "itw_eer"]
shrink = m_eval - m_itw
h4_gaps = by["itw_err_at_dev_thr"] - by["itw_eer"]
verdicts = pd.DataFrame([
    {"hypothesis": "H3 (ITW part)",
     "evidence": f"rung2-rung1 margin: eval {100*m_eval:.1f} pp, ITW {100*m_itw:.1f} pp, shrink {100*shrink:.1f} pp",
     "verdict": "edge shrinks (>= 10 pp)" if shrink >= 0.10 else "no shrinkage of >= 10 pp (not decidable as shrinking)"},
    {"hypothesis": "H4",
     "evidence": "ITW error at dev threshold minus ITW EER: " + ", ".join(f"rung {r}: {100*g:.1f} pp" for r, g in h4_gaps.items()),
     "verdict": "true" if (h4_gaps >= 0.10).sum() >= 2 else "false"},
])
verdicts.to_csv(ROOT / "results/e4_hypotheses.csv", index=False)
verdicts

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(figsize=(6.4, 3.4))
names = ["rung 0\nsilence/duration", "rung 1\nLFCC + gradient boosting", "rung 2\nfrozen XLS-R + LR"]
bars = [("eval_eer", "ASVspoof eval (EER)", "#52514e", None),
        ("itw_eer", "In-the-Wild (EER)", "#e34948", None),
        ("itw_err_at_dev_thr", "In-the-Wild at dev threshold", "#f2a3a2", "////")]
w = 0.26
for j, (col, lab, colr, hatch) in enumerate(bars):
    x = np.arange(3) + (j - 1) * w
    b = ax.bar(x, 100 * e4[col], width=w - 0.02, color=colr, hatch=hatch, edgecolor="white", label=lab)
    for xi, v in zip(x, 100 * e4[col]):
        ax.text(xi, v + 1, f"{v:.0f}", ha="center", fontsize=7.5, color=INK)
ax.axhline(50, color=MUTED, ls=":", lw=1); ax.text(-0.45, 51, "chance (50%)", fontsize=8, color=MUTED)
ax.set_xticks(range(3), names); ax.set_ylim(0, 90); ax.set_ylabel("error %")
ax.grid(axis="y", color=GRID, linewidth=0.6); ax.legend(frameon=False, fontsize=8, loc="upper right")
ax.set_title("Same models, new data source", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/e4_itw.png", dpi=200)
fig.savefig(ROOT / "results/figures/e4_itw.pdf")